# Exercices 7

[Télécharger l'exercice](../07_exercice.zip)

# Exercice : évolution du permafrost au Cervin

Pour un premier exercice en deux dimensions, nous allons modéliser l'évolution de la température au sein du Cervin (coupe nord-sud) sous l'effet du réchauffement climatique. L'augmentation des températures en haute montagne est particulièrement problématique pour la stabilité des pentes et falaises dont la cohésion dépend de la présence de glace. Dans cet exercice, nous allons augmenter la température des flancs du Cervin (auparavant en équilibre thermique) et observer la migration de la limite du permafrost, et de fait l'étendue des pentes déstabilisées.

![](../illu_mod_num_s.png)

## 1. Modèle élémentaire

Avant de s'attaquer au Cervin, nous allons faire un modèle élémentaire dans un simple carré.

Le point clé de ce problème est de résoudre la diffusion dans un domaine en deux dimensions (2D), rectangulaire (Figure ci-dessous). L'évolution du champ de température $T$ qui se diffuse en deux dimensions ($x$ et $y$) est décrite par deux flux ($x$ et $y$) qui se combinent dans le taux de changement :

$$\frac{\partial T}{\partial t} = -\left(\frac{\partial q_x}{\partial x} +\frac{\partial q_y}{\partial y}\right)\:$$

et

$$q_x = - D \frac{\partial T}{\partial x} \: ; \: \: q_y = - D \frac{\partial T}{\partial y}.$$

Nous commençons par modéliser l'évolution du champ de température dans un rectangle dont les paramètres (y compris les conditions initiales et aux bords) sont donnés dans la table ci-dessous. La matrice des températures peut être affichée avec `ax.imshow` comme dans la figure suivante.

![](./fig/initial_rectangle.png)

*L'équilibrage de la diffusion de température en deux dimensions sur un rectangle simple, ici avec des conditions aux bords de 5°C et 15°C respectivement.*

### Paramètres

| **Paramètre**                        | **Valeur** | **Unité** |
|--------------------------------------|------------|-----------|
| Longueur en $x$, $L_x$               | 3000       | m         |
| Longueur en $y$, $L_y$               | 3000       | m         |
| Température initiale                 | 10         | °C        |
| Diffusivité, $D$                     | 3500       | m²/an     |
| Nombre de nœuds $n_x$                | 60         | —         |
| Nombre de nœuds $n_y$                | 70         | —         |
| **modèle élémentaire:**              |            |           |
| Température bord gauche              | 5          | °C        |
| Température bord droite              | 15         | °C        |
| Température bord haut                | 5          | °C        |
| Température bord bas                 | 15         | °C        |
| Durée totale                         | 500        | ans       |

Pour résoudre l'exercice, veuillez utiliser la structure de code suivante. Complétez les paramètres, l'initialisation de la matrice `T`, l'implémentation des flux et de la mise à jour de `T`, ainsi que les conditions aux bords.

```python
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Paramètres physiques
D        = ...  # diffusivité, m2/an
Lx       = ...  # longueur du domaine en x, m
Ly       = ...  # longueur du domaine en y, m
T0       = ...  # température initiale, degC
duree    = ...  # durée totale, ans
T_bas    = ...  # condition aux bords, bas
T_haut   = ...  # condition aux bords, haut
T_gauche = ...  # condition aux bords, gauche
T_droit  = ...  # condition aux bords, droite

# Paramètres numériques
nx          = ...                         # nombre de nœuds en x
ny          = ...                         # nombre de nœuds en y
dx          = Lx / (nx - 1)               # taille d'une cellule en x
dy          = Ly / (ny - 1)               # taille d'une cellule en y
x           = np.linspace(0, Lx, nx)      # coordonnées des nœuds en x
y           = np.linspace(0, Ly, ny)      # coordonnées des nœuds en y
dt_max      = 1.0                         # pas de temps maximal, an
dt_diff     = ...                         # contrainte de la diffusion (en 2D !)
dt          = min(dt_max, dt_diff)        # pas de temps retenu
nt          = ...                         # nombre de pas de temps
freq_affich = 50                          # fréquence d'affichage

# Initialisation : matrice de taille (ny, nx)
T = ...

# création de la figure à l'extérieur de la boucle
fig, ax = plt.subplots()
s    = ax.imshow(T, extent=[0, Lx, 0, Ly], origin='lower', cmap='jet', vmin=5, vmax=15)
cbar = plt.colorbar(s)
cbar.set_label("Température, °C", rotation=270)

# boucle temporelle
for it in range(nt):
    # mise à jour des flux qx et qy en fonction de T
    qx = ...
    qy = ...
    # mise à jour de dTdt (attention aux tailles des matrices)
    dTdt = ...
    # mise à jour de T à l'intérieur du domaine
    T[1:-1, 1:-1] += ...

    # conditions aux bords
    T[0, :]  = ...
    T[-1, :] = ...
    T[:, 0]  = ...
    T[:, -1] = ...

    # affichage
    if it % freq_affich == 0:
        clear_output(wait=True)
        ax.cla()
        ax.imshow(T, extent=[0, Lx, 0, Ly], origin='lower', cmap='jet', vmin=5, vmax=15)
        ax.set_title('Température après ' + str(int(it * dt)) + ' ans')
        ax.set_xlabel('Distance horizontale en x, m')
        ax.set_ylabel('Distance horizontale en y, m')
        display(fig)
```
### ✅ **À vous de faire !**


## 2. Modèle réaliste du Cervin

Afin de modéliser le Cervin simplement, nous réduirons sa géométrie à un carré, qu'il faudra imaginer tourné d'un angle de 45°. Nous reprendrons le code du modèle élémentaire.

![](./fig/matterhorn.png)

*Le Cervin réduit à un carré : à gauche la montagne en coupe, à droite le même domaine redressé, tel qu'il apparaît dans le code. Les deux faces exposées portent les conditions aux bords qui varient le long du bord ; les deux autres restent à 10 °C.*

Nous créons deux vecteurs de conditions aux bords qui représentent les faces nord et sud de la montagne. Le **sommet** correspond au coin supérieur gauche du domaine, en $(x=0,\ y=L_y)$ :

- la **face nord** est le bord supérieur (`T[-1,:]`) : sa température va de $-15$°C au sommet ($x=0$) à $10$°C au pied ($x=L_x$) ;
- la **face sud** est le bord de gauche (`T[:,0]`) : sa température va de $15$°C au pied ($y=0$) à $-5$°C au sommet ($y=L_y$).

Le modèle tournera d'abord pendant 500 ans pour approcher un équilibre thermique. Puis, le réchauffement des faces sud et nord peut commencer (en modifiant les conditions aux bords nord et sud uniformément). Une fois que l'équilibrage thermique fonctionne bien, l'affichage des figures peut être limité à la dernière phase de réchauffement (100 ans au total).

Bien que ce modèle soit très simplifié, il nous permet de constater une dynamique bien réelle dans le massif alpin. En observant le déplacement de la limite du zéro degré Celsius, nous pouvons nous interroger sur le risque lié à la fonte du pergélisol en milieu alpin, et sur ses conséquences pour le paysage et la société.

Les valeurs aux bords et de réchauffement de la Table ci-dessous ne sont que des suggestions, nous pouvons explorer une myriade de scénarios.

| **Paramètre**                        | **Valeur**      | **Unité**  |
|--------------------------------------|-----------------|------------|
| **modèle réaliste:**                 |                 |            |
| Température face nord (de $x=0$ à $x=L_x$) | de -15 à 10 | °C         |
| Température face sud (de $y=0$ à $y=L_y$)  | de 15 à -5  | °C         |
| Température bords restants           | 10              | °C         |
| Durée totale                         | 600             | ans        |
| Début du réchauffement               | 500             | ans        |
| Réchauffement au nord                | +4              | °C/siècle  |
| Réchauffement au sud                 | +8              | °C/siècle  |

> ⚠️ **Attention aux unités !** Les paramètres ne vous sont pas tous donnés dans des unités compatibles entre elles. Convertissez-les explicitement dans votre code, avant la boucle temporelle.



### ✅ **À vous de faire !**